In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install rasterio

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import tensorflow as tf
import rasterio
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
BASE_PATH = "/content/drive/MyDrive/MARIDA"
PATCH_PATH = BASE_PATH + "/patches"

MODEL_PATH = "/content/drive/MyDrive/marine_debris_unet_v3.keras"
DEBRIS_CLASS = 1

In [ ]:
# load with custom objects if needed
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print("Segmentation model loaded!")

Segmentation model loaded!


In [ ]:
def load_patch_by_name(patch_name):
    patch_name = patch_name.replace(".tif", "")
    full_name = "S2_" + patch_name

    for scene in os.listdir(PATCH_PATH):
        scene_path = os.path.join(PATCH_PATH, scene)

        img_path  = os.path.join(scene_path, full_name + ".tif")
        mask_path = os.path.join(scene_path, full_name + "_cl.tif")

        if os.path.exists(img_path) and os.path.exists(mask_path):
            with rasterio.open(img_path) as src:
                img = src.read().astype(np.float32)

            with rasterio.open(mask_path) as src:
                mask = src.read(1)

            return img, mask

    return None, None

In [ ]:
def compute_indices(img):
    img = img / 10000.0

    blue  = img[1]
    green = img[2]
    red   = img[3]
    nir   = img[7]
    swir1 = img[10]
    red_edge2 = img[5]

    ndvi = (nir - red) / (nir + red + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)
    fdi  = nir - (red_edge2 + (swir1 - red_edge2))

    features = np.stack([red, green, blue, nir, ndvi, ndwi, fdi], axis=-1)

    # same normalization as training
    features = (features - np.mean(features)) / (np.std(features) + 1e-6)

    return features.astype(np.float32)

In [ ]:
all_patches = []

for scene in os.listdir(PATCH_PATH):
    for file in os.listdir(os.path.join(PATCH_PATH, scene)):
        if file.endswith(".tif") and not file.endswith("_cl.tif"):
            name = file.replace("S2_", "").replace(".tif", "")
            all_patches.append(name)

print("Total patches:", len(all_patches))

Total patches: 2779


In [ ]:
TH_SEG = 0.15     # segmentation threshold
PIXEL_THRESHOLD = 2  # classification threshold


def classify_from_segmentation(img):
    features = compute_indices(img)

    pred = model.predict(features[np.newaxis,...], verbose=0)[0]

    mask = (pred.squeeze() > TH_SEG)

    pixel_count = np.sum(mask)

    if pixel_count > PIXEL_THRESHOLD:
        return 1, pixel_count
    else:
        return 0, pixel_count

In [ ]:
y_true = []
y_pred = []

for name in tqdm(all_patches):
    img, mask = load_patch_by_name(name)

    if img is None:
        continue

    # ground truth
    true_label = 1 if np.sum(mask == DEBRIS_CLASS) > 0 else 0

    # prediction
    pred_label, pixels = classify_from_segmentation(img)

    y_true.append(true_label)
    y_pred.append(pred_label)

100%|██████████| 2779/2779 [16:51<00:00,  2.75it/s]


In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Confusion Matrix:
[[776 232]
 [108 265]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.77      0.82      1008
           1       0.53      0.71      0.61       373

    accuracy                           0.75      1381
   macro avg       0.71      0.74      0.71      1381
weighted avg       0.78      0.75      0.76      1381



In [ ]:
name = all_patches[45]

img, mask = load_patch_by_name(name)

pred_label, pixels = classify_from_segmentation(img)

print("Predicted:", "Debris" if pred_label else "No Debris")
print("Pixel count:", pixels)

Predicted: Debris
Pixel count: 24


In [ ]:
start = 0
end = 100   # change range as needed

for i in range(start, end):
    name = all_patches[i]

    img, _ = load_patch_by_name(name)

    if img is None:
        continue

    pred_label, pixels = classify_from_segmentation(img)

    print(f"Image {i} ({name}) →",
          "Debris" if pred_label else "No Debris",
          "| Pixels:", pixels)

Image 0 (8-3-18_16QED_14) → No Debris | Pixels: 0
Image 2 (8-3-18_16QED_10) → No Debris | Pixels: 0
Image 3 (8-3-18_16QED_12) → No Debris | Pixels: 0
Image 4 (8-3-18_16QED_13) → No Debris | Pixels: 1
Image 5 (8-3-18_16QED_16) → No Debris | Pixels: 0
Image 8 (8-3-18_16QED_0) → No Debris | Pixels: 0
Image 11 (8-3-18_16QED_15) → No Debris | Pixels: 0
Image 12 (8-3-18_16QED_11) → No Debris | Pixels: 0
Image 14 (8-3-18_16QED_1) → Debris | Pixels: 5
Image 19 (8-3-18_16QED_8) → No Debris | Pixels: 2
Image 20 (8-3-18_16QED_9) → No Debris | Pixels: 0
Image 21 (8-3-18_16QED_17) → No Debris | Pixels: 0
Image 22 (8-3-18_16QED_7) → No Debris | Pixels: 0
Image 24 (8-3-18_16QED_18) → No Debris | Pixels: 0
Image 27 (8-3-18_16QED_6) → Debris | Pixels: 3
Image 30 (8-3-18_16QED_2) → Debris | Pixels: 3
Image 31 (8-3-18_16QED_4) → No Debris | Pixels: 2
Image 36 (8-3-18_16QED_5) → No Debris | Pixels: 0
Image 37 (8-3-18_16QED_3) → No Debris | Pixels: 0
Image 39 (9-10-17_16PEC_0) → No Debris | Pixels: 1
Image